In [1]:
# relevant import

## IMPORTS

In [2]:
import importlib
import Diffusion
import LossFunctions
import ConsistencyModels
import  FlowMatching
from ConsistencyModels import ConsistencyModel
import dist_utils
import Optimization

importlib.reload(Diffusion)
importlib.reload(LossFunctions)
importlib.reload(ConsistencyModels)
importlib.reload(FlowMatching)
importlib.reload(dist_utils)
importlib.reload(Optimization)



<module 'Optimization' from 'C:\\Users\\orine\\PycharmProjects\\finalThesis\\Optimization.py'>

In [3]:
import matplotlib.pyplot as plt
import time
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

In [4]:
def get_param_mog_with_target(dim_data=3,num_components=2):

    # generate gmm parameters
    mu_list, Sigma_list, alpha=dist_utils.generate_gmm_parameters(d=dim_data, num_components=num_components,seed=1)

    # find the x_star
    samples = dist_utils.generate_mog_samples(150, mu_list, Sigma_list, alpha).float().to(device)
    idx = torch.randint(0, 150, (1,))
    x_star = samples[idx, :dim_data-1]

    # compute the Y|X=x^*
    mog_means, mog_variances = dist_utils.compute_conditionals(mu_list, Sigma_list, x_star.reshape(-1,1))
    weights = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star.reshape(-1,1))

    return mu_list, Sigma_list, alpha,mog_means, mog_variances,weights,x_star

In [5]:
def generate_data(dim_data, n_sample):
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = get_param_mog_with_target(dim_data=dim_data, num_components=2)
    X = dist_utils.generate_mog_samples(n_sample, mu_list, Sigma_list, alpha).float().to(device)
    return X, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights
def train_diffusion_models(X, n_epochs, nn_n_units, nn_depth, condition_on,diffusion_steps):
    start = time.time()
    model_cond = Diffusion.DiffusionModel(
        nfeatures=X.shape[1], nblocks=nn_depth, nunits=nn_n_units,
        condition=True, condition_on=condition_on, diffusion_steps=diffusion_steps
    )
    model_cond.train_model(X, nepochs=n_epochs, batch_size=3*2048, condition_on=condition_on)

    X_cond = X[:, :condition_on]
    model_uncond = Diffusion.DiffusionModel(
        nfeatures=X_cond.shape[1], nblocks=nn_depth, nunits=nn_n_units,
        condition=False, diffusion_steps=diffusion_steps
    )
    model_uncond.train_model(X_cond, nepochs=n_epochs, batch_size=3*2048, condition_on=condition_on)

    return model_cond, model_uncond, time.time() - start
def train_consistency_model(X, condition_on, nn_n_units, n_epochs):
    start = time.time()
    model = ConsistencyModel(nfeatures=X.shape[1] - condition_on, condition_on=condition_on, nunits=nn_n_units)
    model.train_model(X, nepochs=n_epochs, batch_size=256, device="cpu", condition=condition_on)
    return model, time.time() - start


def train_flow_models(X, mog_means, y_dim, nn_n_units, nn_depth, n_epochs):
    start = time.time()
    input_dim = mog_means[0].shape[0]
    vf_cond = FlowMatching.FMModel(input_dim, y_dim, nn_n_units, nn_depth, device=device)
    vf_cond.train_FM(lr=0.001, batch_size=3*2048, X=X, nepochs=n_epochs)

    vf_uncond = FlowMatching.FMModel(y_dim, 0, nn_n_units, nn_depth, device=device)
    vf_uncond.train_FM(lr=0.001, batch_size=3*2048, X=X[:, :y_dim], nepochs=n_epochs)

    return vf_cond, vf_uncond, time.time() - start
def run_optimizations(models, mog_data, condition_on,device):
    model_uncond, model_cond, model_consistency, vf_cond, vf_uncond = models
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights = mog_data
    results = []

    def record(method_name, x_optim, start_time):
        l1 = dist_utils.warpper_L1_distance(x_optim, mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
        results.append({"method": method_name, "l1_distance": l1, "opt_time": time.time() - start_time})

    # Diffusion + LGD
    start = time.time()
    _,x,_ = Optimization.optimize_LGD(model_uncond, model_cond, mog_means, mog_variances, weights, mu_list, Sigma_list, alpha, nsamples=250, loss="MMD",device=device)
    record("LGD", x.reshape(-1, 1), start)

    # Diffusion + DPS
    start = time.time()
    _,x,_  = Optimization.optimize_DPS(model_uncond, model_cond, mog_means, mog_variances, weights, mu_list, Sigma_list, alpha, nsamples=500, loss="MMD",device=device)
    record("DPS", x.reshape(-1, 1), start)

    # Consistency + LGD
    start = time.time()
    _,x,_  = Optimization.optimize_LGD(model_uncond, model_consistency, mog_means, mog_variances, weights, mu_list, Sigma_list, alpha, nsamples=250, loss="MMD", CM=True,device=device)
    record("LGD_CM", x.reshape(-1, 1), start)

    # Consistency + DPS
    start = time.time()
    _,x,_  = Optimization.optimize_DPS(model_uncond, model_consistency, mog_means, mog_variances, weights, mu_list, Sigma_list, alpha, nsamples=250, loss="MMD", CM=True,device=device)
    record("DPS_CM", x.reshape(-1, 1), start)

    # Flow Matching + DFLOW (MMD)
    start = time.time()
    x, _ = Optimization.optimize_DFLOW(vf_cond, vf_uncond, device, mog_means, mog_variances, weights, max_iter=100, FLAG=False, n_sample=100, loss_method="MMD")
    record("DFLOW_MMD", x.reshape(-1, 1), start)

    # Flow Matching + DFLOW (KL)
    start = time.time()
    x, _ = Optimization.optimize_DFLOW(vf_cond, vf_uncond, device, mog_means, mog_variances, weights, max_iter=100, FLAG=False, n_sample=50, loss_method="KL",)
    record("DFLOW_KL", x.reshape(-1, 1), start)

    return results

import torch
import os
from datetime import datetime

def save_model(model, name, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(save_dir, f"{name}_{timestamp}.pt")
    torch.save(model.state_dict(), path)
    print(f"Saved model: {path}")
    return path

def run_optimizations_n_times(n_times, dim_data, n_sample, n_epochs, nn_n_units, nn_depth,diffusion_steps,device):
    # Generate data and train once
    X, *mog_data = generate_data(dim_data, n_sample)
    condition_on = mog_data[0][0].shape[0] - mog_data[3][0].shape[0]

    # Train models
    model_cond, model_uncond, train_time_diff = train_diffusion_models(X, n_epochs, nn_n_units, nn_depth, condition_on,diffusion_steps)
    model_consistency, train_time_cons = train_consistency_model(X, condition_on, nn_n_units, n_epochs)
    vf_cond, vf_uncond, train_time_flow = train_flow_models(X, mog_data[3], condition_on, nn_n_units, nn_depth, n_epochs)

    # Save models
    save_model(model_uncond, "model_uncond")
    save_model(model_cond, "model_cond")
    save_model(model_consistency, "model_consistency")
    save_model(vf_cond, "vf_cond")
    save_model(vf_uncond, "vf_uncond")

    models = (model_uncond, model_cond, model_consistency, vf_cond, vf_uncond)

    # Run optimizations multiple times
    all_results = []
    for _ in range(n_times):
        optimization_results = run_optimizations(models, mog_data, condition_on,device)

        for r in optimization_results:
            if r['method'].startswith("DFLOW"):
                r["train_time"] = train_time_flow
            elif r['method'].endswith("CM"):
                r["train_time"] = train_time_cons + train_time_diff
            else:
                r["train_time"] = train_time_diff

        all_results.append(optimization_results)

    return all_results,*mog_data,models




In [6]:
# @title
if torch.cuda.is_available():
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

Using cpu.


In [7]:
all_results,*mog_data,models = run_optimizations_n_times(5,3,50_000, 1, 256,4,diffusion_steps=100,device=device)

C:\Users\orine\PycharmProjects\finalThesis\dist_utils.py:403: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4416.)
  exponent = -0.5 * diff.T @ Sigma_22_inv @ diff
loss: 8.673036: 100%|██████████| 1/1 [00:04<00:00,  4.04s/it]


Saved model: saved_models\model_uncond_20250517_165437.pt
Saved model: saved_models\model_cond_20250517_165437.pt
Saved model: saved_models\model_consistency_20250517_165437.pt
Saved model: saved_models\vf_cond_20250517_165437.pt
Saved model: saved_models\vf_uncond_20250517_165437.pt


Step 1 | pred_x0_model: tensor([[  1.7901, -82.5575]], grad_fn=<AddBackward0>) | pred_x0: tensor([[  1.7510, -82.5526]], grad_fn=<AddBackward0>) | batch_loss: 1.625: 100%|██████████| 99/99 [14:14<00:00,  8.63s/it] 
100%|██████████| 7/7 [00:00<00:00, 198.09it/s]
Step 1 | pred_x0 loss: 1.530: 100%|██████████| 99/99 [06:13<00:00,  3.77s/it] 
100%|██████████| 7/7 [00:00<00:00, 393.79it/s]
Step 1 | pred_x0_model: tensor([[ -6.7310, -30.1565]], grad_fn=<AddBackward0>) | pred_x0: tensor([[ -6.7271, -30.1580]], grad_fn=<AddBackward0>) | batch_loss: 1.238: 100%|██████████| 99/99 [01:07<00:00,  1.46it/s] 
100%|██████████| 7/7 [00:00<00:00, 257.52it/s]
Step 1 | pred_x0 loss: 1.255: 100%|██████████| 99/99 [00:23<00:00,  4.14it/s] 
100%|██████████| 7/7 [00:00<00:00, 176.16it/s]
Step 38 | pred_x0_model: tensor([[-12.0067, -61.4069]], grad_fn=<AddBackward0>) | pred_x0: tensor([[-10.7131, -52.1054]], grad_fn=<AddBackward0>) | batch_loss: 1.603:  62%|██████▏   | 61/99 [09:12<05:44,  9.06s/it]


KeyboardInterrupt: 

In [ ]:
STOP

In [7]:
# # 1D COND 1D
# mu_list = [np.array([-5, 4]),
#            np.array([5, -4]),
#            np.array([3, 5]),np.array([-1, 5]), np.array([0, 0]), np.array([5, -1]),np.array([-5, -1]), np.array([2, 1]),np.array([1, -7])
#            ]
#
# point_l = [[int(p[1]), int(p[0])] for p in mu_list]
#
# mu_list = [torch.tensor(p, dtype=torch.float64) for p in point_l]
#
# # Convert Sigma_list to 3D covariance matrices
# Sigma_list = [
#     torch.tensor([[0.5,0.25],[0.12,0.2]], dtype=torch.float64)
# ] * len(mu_list)
#
# # Mixture weights
# alpha =torch.tensor( [1 / len(mu_list)] * len(mu_list),dtype=torch.float64)
# # Generate samples
# device = 'cpu'  # Change to 'cuda' if using a GPU
# # X = dist_utils.generate_mog_samples(n_samples, mu_list, Sigma_list, alpha )
#
#
# mog_means = [
#     torch.tensor([5.0], dtype=torch.float64),
#     # torch.tensor([-5.0], dtype=torch.float64),
#
# ]
# mog_variances = [torch.tensor([.14], dtype=torch.float64),
#      # torch.tensor([.25], dtype=torch.float64),
#      ]
# weights = torch.tensor([1.,
#                         # .5
#                         ], dtype=torch.float64)


In [8]:
# # 2D COND 1D
#
# # Means (3D vectors)
# mu_list = [
#     torch.tensor([-5.0, -4.0, 3.0], dtype=torch.float64), # First component
# torch.tensor([1.0, 3.5, 4.5], dtype=torch.float64), # Second component
#  torch.tensor([3.0, 1.5, 8.5], dtype=torch.float64), # Second component
#  torch.tensor([5.0, 4.0, 5.0], dtype=torch.float64), # Second component
#  torch.tensor([7.0, -1.0, -8.0], dtype=torch.float64) # Second component
# ]
#
# # Covariance matrices (3x3)
# Sigma_list = [
#  torch.tensor([[.2, 0.1, 0.0], [0.1, 0.5, 0.0], [0.0, 0.0, 0.3]], dtype=torch.float64),
#  torch.tensor([[.2, 0.1, 0.0], [0.1, 0.5, 0.0], [0.0, 0.0, 0.3]], dtype=torch.float64),
#  torch.tensor([[.2, 0.1, 0.0], [0.1, 0.5, 0.0], [0.0, 0.0, 0.3]], dtype=torch.float64),
#  torch.tensor([[.2, 0.1, 0.0], [0.1, .5, 0.0], [0.0, 0.0, 0.3]], dtype=torch.float64),
# torch.tensor([[.2, 0.1, 0.0], [0.1, 0.5, 0.0], [0.0, 0.0, 0.3]], dtype=torch.float64),
# ]
#
# # Mixture weights (probability of choosing each component)
# alpha = torch.tensor([1., 1.,1.,1.,1.], dtype=torch.float64)*1/len(Sigma_list)
#
# # Means (3D vectors)
# mog_means = [
#  torch.tensor([4., 5.001], dtype=torch.float64),# First component
#  # torch.tensor([4.1, 5.1], dtype=torch.float64),
# ]
#
# # Covariance matrices (3x3)
# mog_variances = [
#     torch.tensor([[ .45, 0.0], [0.0, 0.3]], dtype=torch.float64),
# ]
#
# # Mixture weights (probability of choosing each component)
# weights = torch.tensor([1.], dtype=torch.float64)
#


In [9]:
# ## 4D COND 2D
# # Means (3D vectors)
# mu_list = [
#     torch.tensor([-4.,-5.0, -4.0, 3.0], dtype=torch.float64),  # First component
#     torch.tensor([1.0,1.0, 3.5, 4.5], dtype=torch.float64),  # Second component
#     torch.tensor([3.,3.0, 1.5, 8.5], dtype=torch.float64),  # Second component
#     torch.tensor([4.,5.0, 4.0, 5.0], dtype=torch.float64),  # Second component
#     torch.tensor([7.,7.0, -1.0, -8.0], dtype=torch.float64)  # Second component
# ]
#
# # Covariance matrices (3x3)
# Sigma_list = [
#     torch.tensor([[.2, 0.1, 0.0, 0.0], [0.1, 0.5, 0.0, 0.0], [0.0, 0.0, 0.3, 0.0], [0.0, 0.0, 0.0, 0.4]], dtype=torch.float64),
#     torch.tensor([[.2, 0.1, 0.0, 0.0], [0.1, 0.5, 0.0, 0.0], [0.0, 0.0, 0.3, 0.0], [0.0, 0.0, 0.0, 0.4]], dtype=torch.float64),
#     torch.tensor([[.2, 0.1, 0.0, 0.0], [0.1, 0.5, 0.0, 0.0], [0.0, 0.0, 0.3, 0.0], [0.0, 0.0, 0.0, 0.4]], dtype=torch.float64),
#     torch.tensor([[.2, 0.1, 0.0, 0.0], [0.1, 0.5, 0.0, 0.0], [0.0, 0.0, 0.3, 0.0], [0.0, 0.0, 0.0, 0.4]], dtype=torch.float64),
#     torch.tensor([[.2, 0.1, 0.0, 0.0], [0.1, 0.5, 0.0, 0.0], [0.0, 0.0, 0.3, 0.0], [0.0, 0.0, 0.0, 0.4]], dtype=torch.float64)
# ]
#
# # Mixture weights (probability of choosing each component)
# alpha = torch.tensor([1., 1.,1.,1.,1.], dtype=torch.float64)*1/len(Sigma_list)
#
# # Means (3D vectors)
# mog_means = [
#     torch.tensor([4.0, 5.0], dtype=torch.float64),  # First component
#     # torch.tensor([4.1, 5.1], dtype=torch.float64),
# ]
#
# # Covariance matrices (3x3)
# mog_variances = [
#     torch.tensor([[ 0.5, 0.0], [0.0, 0.3]], dtype=torch.float64),
# ]
#
# # Mixture weights (probability of choosing each component)
# weights = torch.tensor([1.], dtype=torch.float64)


In [11]:
d = 3  # Number of dimensions
num_components =2  # Number of Gaussian components

mu_list, Sigma_list, alpha=dist_utils.generate_gmm_parameters(d=d, num_components=num_components,seed=1)
conditional = dist_utils.generate_mog_samples(1, mu_list, Sigma_list, alpha).float().to(device)

samples = dist_utils.generate_mog_samples(150, mu_list, Sigma_list, alpha).float().to(device)
idx = torch.randint(0, 150, (1,))
x_star = samples[idx, :d-1]

mog_means, mog_variances = dist_utils.compute_conditionals(mu_list, Sigma_list, x_star.reshape(-1,1))
weights = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star.reshape(-1,1))

C:\Users\orine\PycharmProjects\finalThesis\dist_utils.py:403: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4416.)
  exponent = -0.5 * diff.T @ Sigma_22_inv @ diff


In [42]:
l1_distance = dist_utils.warpper_L1_distance(x_star.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(l1_distance)

tensor([[2.2056, 2.7522]])

In [13]:
X = dist_utils.generate_mog_samples(50_000, mu_list, Sigma_list, alpha).float().to(device)

In [14]:
print(X)

tensor([[ 1.8604,  1.6761,  1.0694],
        [ 2.3957,  0.1524, -0.5623],
        [ 1.4988,  1.2033,  1.6972],
        ...,
        [ 2.4716, -2.5005, -3.2914],
        [ 2.3260,  0.2409, -0.3186],
        [ 0.7681,  4.4305,  3.3576]])


## LGD

In [16]:
# better version
# init a model, train
nfeatures= X.shape[1]
condition_on=mu_list[0].shape[0]-mog_means[0].shape[0]
model_cond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=4, nunits=128, condition=True,condition_on=condition_on,diffusion_steps=100)
losses = model_cond.train_model(X, nepochs=300, batch_size=3*2048,condition_on=condition_on)

loss: 0.178001: 100%|██████████| 300/300 [09:39<00:00,  1.93s/it]


In [17]:
# init a model, train
X_for_cond_only=X[:,:model_cond.condition_on]
nfeatures= X_for_cond_only.shape[1]
model_uncond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=4, nunits=128, condition=False,diffusion_steps=100)
losses = model_uncond.train_model(X_for_cond_only, nepochs=300, batch_size=3*2048,condition_on=condition_on)

loss: 0.449513: 100%|██████████| 300/300 [09:38<00:00,  1.93s/it]


### with normal diffusion prediction

In [18]:
best_x_t, best_x_0_cont_xt_hat, best_mmd_loss = Optimization.optimize_LGD(model_uncond, model_cond, mog_means, mog_variances, weights,mu_list, Sigma_list,alpha, nsamples=250, loss="MMD")
l1_distance = dist_utils.warpper_L1_distance(best_x_t,mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(f"best l1 of best_x_t:{l1_distance}")

l1_distance = dist_utils.warpper_L1_distance(best_x_0_cont_xt_hat.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(f"best l1 of best_x_0_cont_xt_hat:{l1_distance}")

Step 1 | pred_x0_model: tensor([[0.7877, 3.2097]], grad_fn=<AddBackward0>)| pred_x0: tensor([[0.7861, 3.1733]], grad_fn=<AddBackward0>)| mean_loss: 0.762: 100%|██████████| 99/99 [09:44<00:00,  5.90s/it]     


tensor([[1.9948],
        [3.5159]])


 71%|███████▏  | 5/7 [00:00<00:00, 217.85it/s]

tensor(1.3825, dtype=torch.float64)


In [19]:
best_x_t, best_pred_x0, best_mmd_loss= Optimization.optimize_DPS(model_uncond, model_cond, mog_means, mog_variances, weights,mu_list, Sigma_list,alpha, nsamples=500, loss="MMD")

l1_distance = dist_utils.warpper_L1_distance(best_x_t,mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(f"best l1 of best_x_t:{l1_distance}")

l1_distance = dist_utils.warpper_L1_distance(best_pred_x0.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(f"best l1 of best_x_0_cont_xt_hat:{l1_distance}")

Step 1 | pred_x0: tensor([[ 1.1481, -0.1892]], grad_fn=<AddBackward0>)| mean_loss: 1.311: 100%|██████████| 99/99 [03:30<00:00,  2.12s/it] 


tensor([[ 1.1486],
        [-0.1894]])


100%|██████████| 7/7 [00:00<00:00, 279.55it/s]

tensor(1.7352, dtype=torch.float64)


### with CM prediction

In [20]:
B, C = X.shape
nfeatures = C - condition_on

Cos_model = ConsistencyModel(nfeatures=nfeatures, condition_on=condition_on, nunits=128)
Cos_model.train_model(X=X, nepochs= 500,batch_size=256, device= "cpu", condition=condition_on)

loss: 0.000678, mu: 0.9993: 100%|██████████| 500/500 [1:00:13<00:00,  7.23s/it]


In [21]:
# xh,_=Cos_model.sample(nsamples=250,condition_x=torch.tensor([4.,5.0]))
# # Convert to numpy
# xh_cpu = xh.detach().cpu().numpy()
#
# # Scatter plot of first column vs. second column
# plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.6, s=20)
# plt.title("Scatter Plot of First vs. Second Columns of xh")
# plt.xlabel("First Column")
# plt.ylabel("Second Column")
# plt.grid(True)
# plt.show()

In [83]:
best_x_t, best_x_0_cont_xt_hat, best_mmd_loss = Optimization.optimize_LGD(model_uncond, Cos_model, mog_means, mog_variances, weights,mu_list, Sigma_list,alpha, nsamples=500, loss="MMD",CM=True)
l1_distance = dist_utils.warpper_L1_distance(best_x_t.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(f"best l1 of best_x_t:{l1_distance}")
l1_distance = dist_utils.warpper_L1_distance(best_x_0_cont_xt_hat.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(f"best l1 of best_x_0_cont_xt_hat:{l1_distance}")


Step 1 | pred_x0_model: tensor([[-2.2476,  0.8547]], grad_fn=<AddBackward0>) | pred_x0: tensor([[-2.2126,  0.9231]], grad_fn=<AddBackward0>) | batch_loss: 1.273: 100%|██████████| 99/99 [01:32<00:00,  1.07it/s] 
100%|██████████| 7/7 [00:00<00:00, 365.18it/s]


best l1 of best_x_t:1.731898565306649


100%|██████████| 7/7 [00:00<00:00, 392.75it/s]

best l1 of best_x_0_cont_xt_hat:1.7176583061471686


In [80]:
best_x_t, best_pred_x0, best_mmd_loss = Optimization.optimize_DPS(model_uncond, Cos_model, mog_means, mog_variances, weights,mu_list, Sigma_list,alpha, nsamples=250, loss="MMD",CM=True)
l1_distance = dist_utils.warpper_L1_distance(best_x_t.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(f"best l1 of best_x_t:{l1_distance}")
l1_distance = dist_utils.warpper_L1_distance(best_pred_x0.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(l1_distance)

Step 1 | pred_x0: tensor([[ 1.3837, -0.9780]], grad_fn=<AddBackward0>) | mean_loss: 0.568: 100%|██████████| 99/99 [00:15<00:00,  6.48it/s] 


tensor([[ 1.3941, -0.9454]])


100%|██████████| 7/7 [00:00<00:00, 370.06it/s]

tensor(1.6008, dtype=torch.float64)


## D-FLOW

In [24]:
input_dim =mog_means[0].shape[0]
y_dim = mu_list[0].shape[0]-mog_means[0].shape[0]
hidden_dim = 256
depth = 4


In [25]:

vf_y_cond_x = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=y_dim,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
vf_y_cond_x.train_FM(lr=0.001, batch_size=1024, X=X, nepochs=100)


loss: 2.127974: 100%|██████████| 100/100 [07:39<00:00,  4.59s/it]


In [26]:
first_column_x=X[:,:y_dim]#print()#print()
input_dim = first_column_x.shape[1]
vf_X = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=0,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
vf_X.train_FM(lr=0.001, batch_size=1024, X=first_column_x, nepochs=100)


loss: 3.277082: 100%|██████████| 100/100 [06:37<00:00,  3.97s/it]


### with loss of samples

In [31]:
x_optim,loss= Optimization.optimize_DFLOW(vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights, max_iter=100,FLAG=True,n_sample=1_00,loss_method="MMD")
print(x_optim)
l1_distance = dist_utils.warpper_L1_distance(x_optim.reshape(-1,1), mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(l1_distance)

x_0: [[0.23567963 0.05838581]] | x_1: tensor([[2.0386, 0.3327]], grad_fn=<SelectBackward0>) | loss: 1.704146315388035
x_0: [[-0.02994215  0.792764  ]] | x_1: tensor([[1.7195, 1.6019]], grad_fn=<SelectBackward0>) | loss: 0.7960595353565192
x_0: [[-2.4408026  9.481123 ]] | x_1: tensor([[-2.0233, 17.4349]], grad_fn=<SelectBackward0>) | loss: 1.1277135577736725
x_0: [[-0.7889949  3.5282698]] | x_1: tensor([[-0.2192, 11.2043]], grad_fn=<SelectBackward0>) | loss: 1.1035261889503198
x_0: [[-0.23747338  1.5406736 ]] | x_1: tensor([[1.2346, 3.4406]], grad_fn=<SelectBackward0>) | loss: 1.03049938612748
x_0: [[-0.1050615  1.0634823]] | x_1: tensor([[1.5875, 2.1118]], grad_fn=<SelectBackward0>) | loss: 0.4680650447642307
x_0: [[-0.07961796  1.2559012 ]] | x_1: tensor([[1.5545, 2.5378]], grad_fn=<SelectBackward0>) | loss: 0.46832515845212175
x_0: [[-0.09490632  1.1402818 ]] | x_1: tensor([[1.5790, 2.2732]], grad_fn=<SelectBackward0>) | loss: 0.6901827854827265
x_0: [[-0.10467597  1.0663979 ]] | x_1

 86%|████████▌ | 6/7 [00:00<00:00, 277.25it/s]

tensor(1.3326, dtype=torch.float64, grad_fn=<MulBackward0>)


### with KL loss

In [41]:
x_optim,loss= Optimization.optimize_DFLOW(vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights, max_iter=100,FLAG=True,n_sample=50,loss_method="KL")
print(x_optim)
l1_distance = dist_utils.warpper_L1_distance(x_optim.reshape(-1,1),mu_list, Sigma_list, alpha, mog_means, mog_variances, weights)
print(l1_distance)

x_0: [[-1.2033242  0.8817902]] | x_1: tensor([[0.4126, 1.6174]], grad_fn=<SelectBackward0>) | loss: 1.4646204710006714
x_0: [[-1.2044511  1.8806634]] | x_1: tensor([[0.4484, 5.1385]], grad_fn=<SelectBackward0>) | loss: 0.83070307970047
x_0: [[-1.2436622  1.8760761]] | x_1: tensor([[0.4292, 5.1262]], grad_fn=<SelectBackward0>) | loss: 1.1269493103027344
x_0: [[-1.2046965  1.8806347]] | x_1: tensor([[0.4483, 5.1384]], grad_fn=<SelectBackward0>) | loss: 1.4599260091781616
x_0: [[-1.2044756  1.8806605]] | x_1: tensor([[0.4484, 5.1385]], grad_fn=<SelectBackward0>) | loss: 1.5614019632339478
x_0: [[-1.2044511  1.8806634]] | x_1: tensor([[0.4484, 5.1385]], grad_fn=<SelectBackward0>) | loss: 1.0752549171447754
tensor([[0.4484, 5.1385]], grad_fn=<SelectBackward0>)


100%|██████████| 7/7 [00:00<00:00, 177.71it/s]

tensor(1.5071, dtype=torch.float64, grad_fn=<MulBackward0>)
